<a href="https://colab.research.google.com/github/murillo-borges/webscraping-livelo-esfera/blob/main/Webscraping_Whats_Milhas_New_Version.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Dados padronizados

### Instalando Bibliotecas


In [1]:
pip install pandas

In [2]:
pip install datetime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.7/270.7 kB 14.4 MB/s eta 0:00:00


### Importando Bibliotecas


In [3]:
import requests
import pandas as pd
import re
import os
import datetime
import time
from bs4 import BeautifulSoup
from zoneinfo import ZoneInfo
import textwrap

### Modo de Teste no Colab (upload das credenciais de teste)

In [4]:
# =========================================
# 🧪 MODO DE TESTE NO COLAB
# =========================================
# Detecta automaticamente se está rodando no Google Colab. Se estiver,
# pede upload dos arquivos de credenciais de TESTE e os injeta em
# os.environ — assim todo o resto do notebook (que já lê tudo via
# os.environ[...]) funciona sem precisar alterar mais nenhuma célula.
#
# No GitHub Actions o import do google.colab falha (pacote não existe
# nesse ambiente), então esse bloco é pulado automaticamente e as
# variáveis de ambiente reais (secrets do GitHub) são usadas normalmente.
# Ou seja: não precisa lembrar de "reverter" nada antes de subir pro ar.

try:
    from google.colab import files
    RODANDO_NO_COLAB = True
except ImportError:
    RODANDO_NO_COLAB = False

# Coloque True quando precisar subir os arquivos de novo (ex: trocou uma
# chave/credencial e o arquivo antigo ainda está em cache na sessão do
# Colab). Depois de subir o que precisava, pode voltar pra False.
FORCAR_REUPLOAD = True

if RODANDO_NO_COLAB:
    arquivos_necessarios = [
        "id_grupo_teste.txt",
        "instance_id.txt",
        "api_key.txt",
        "sheet_id.txt",
        "SHEET_ID_CONTROLE_ENVIOS.txt",
        "gemini_key.txt",
        "url_scraping.txt",
        "url_scraping_two.txt",
        "credenciais.json",
    ]

    if FORCAR_REUPLOAD:
        faltando = arquivos_necessarios
    else:
        faltando = [a for a in arquivos_necessarios if not os.path.exists(a)]

    if faltando:
        print(f"📤 Selecione estes arquivos no seletor abaixo (pode escolher só os que mudaram): {faltando}")
        files.upload()

    def ler_arquivo_teste(nome_arquivo):
        with open(nome_arquivo, "r") as f:
            return f.read().strip()

    os.environ["ID_GRUPO_PESSOAL_TESTES"]  = ler_arquivo_teste("id_grupo_teste.txt")
    os.environ["INSTANCE_ID"]              = ler_arquivo_teste("instance_id.txt")
    os.environ["API_KEY"]                  = ler_arquivo_teste("api_key.txt")
    os.environ["SHEET_ID"]                 = ler_arquivo_teste("sheet_id.txt")
    os.environ["SHEET_ID_CONTROLE_ENVIOS"] = ler_arquivo_teste("SHEET_ID_CONTROLE_ENVIOS.txt")
    os.environ["GEMINI_API_KEY"]           = ler_arquivo_teste("gemini_key.txt")
    os.environ["URL_SCRAPING"]             = ler_arquivo_teste("url_scraping.txt")
    os.environ["URL_SCRAPING_TWO"]         = ler_arquivo_teste("url_scraping_two.txt")
    os.environ["GOOGLE_CREDENTIALS"]       = ler_arquivo_teste("credenciais.json")

    print("✅ Variáveis de ambiente de teste carregadas a partir dos arquivos enviados no Colab.")
else:
    print("➡️ Não estamos no Colab — usando as variáveis de ambiente do GitHub Actions normalmente.")

📤 Selecione estes arquivos no seletor abaixo (pode escolher só os que mudaram): ['id_grupo_teste.txt', 'instance_id.txt', 'api_key.txt', 'sheet_id.txt', 'SHEET_ID_CONTROLE_ENVIOS.txt', 'gemini_key.txt', 'url_scraping.txt', 'url_scraping_two.txt', 'credenciais.json']


Saving gemini_key.txt to gemini_key.txt
Saving SHEET_ID_CONTROLE_ENVIOS.txt to SHEET_ID_CONTROLE_ENVIOS.txt
Saving api_key.txt to api_key.txt
Saving url_scraping.txt to url_scraping.txt
Saving credenciais.json to credenciais.json
Saving openai_key.txt to openai_key.txt
Saving id_grupo_teste.txt to id_grupo_teste.txt
Saving url_scraping_two.txt to url_scraping_two.txt
Saving instance_id.txt to instance_id.txt
Saving id_grupo.txt to id_grupo.txt
Saving sheet_id.txt to sheet_id.txt
✅ Variáveis de ambiente de teste carregadas a partir dos arquivos enviados no Colab.


### Definindo o ID do Grupo

In [5]:
# 🚦 FLAG ÚNICA: para onde as mensagens são enviadas.
# True  -> grupo de TESTES (variável de ambiente ID_GRUPO_PESSOAL_TESTES)
# False -> grupo REAL (variável de ambiente ID_GRUPO_MILHAS)
#
# Essa é a única coisa que você precisa mudar para trocar entre testar e
# publicar de verdade. Antes de deixar o GitHub Actions rodar sozinho nos
# horários automáticos, confirme que está False.
ENVIAR_PARA_GRUPO_TESTE = True

if ENVIAR_PARA_GRUPO_TESTE:
    id_grupo_envio = os.environ['ID_GRUPO_PESSOAL_TESTES']
    print("🧪 ENVIAR_PARA_GRUPO_TESTE = True -> enviando para o GRUPO DE TESTES")
else:
    id_grupo_envio = os.environ['ID_GRUPO_MILHAS']
    print("🚀 ENVIAR_PARA_GRUPO_TESTE = False -> enviando para o GRUPO REAL")

🧪 ENVIAR_PARA_GRUPO_TESTE = True -> enviando para o GRUPO DE TESTES


### Definindo os dados da API para envio no Whatsapp

In [6]:
INSTANCE_ID = os.environ['INSTANCE_ID']
API_KEY = os.environ['API_KEY']

# Endpoint
url = f"https://api.zapperapi.com/{INSTANCE_ID}/messages/text" #url para mensagens de texto
url_media = f"https://api.zapperapi.com/{INSTANCE_ID}/messages/media" #url para mensagens de texto com imagens

headers = {
    "X-Api-Key": API_KEY,
    "Content-Type": "application/json"
}

### Função para Webscrapping

In [7]:
#Coletando a data do dia
data_hoje = datetime.datetime.now(ZoneInfo("America/Sao_Paulo")).date()
data_formatada = data_hoje.strftime("%d/%m/%Y")

# Lista de nomes que você quer capturar (normalizados)
nomes_alvo = ["livelo", "shopping livelo", "esfera", "smiles", "azul", "shopping latam", "latam", "pass", "LATAM Pass", "Shopping Livelo"]

#Buscando a hora atual
agora_brasilia = datetime.datetime.now(ZoneInfo("America/Sao_Paulo"))

hora_atual = int(agora_brasilia.strftime("%H"))

def coletar_promocoes(url, parceiro, data_formatada, nomes_alvo):
    pagina = requests.get(url)
    dados_pagina = BeautifulSoup(pagina.text, 'html.parser')

    # --- Captura dos nomes ---
    list_nomes_clubes = []
    for sp in dados_pagina.select("div.d-grid span"):
        nome = sp.get_text(strip=True)
        nome_norm = nome.lower()

        if nome_norm not in nomes_alvo:
            continue

        # Emojis
        if nome_norm in("livelo", "Shopping Livelo"):
            nome = f"🟣 {nome}"
        elif nome_norm == "esfera":
            nome = f"🔴 {nome}"
        elif nome_norm == "smiles":
            nome = f"🟠 {nome}"
        elif nome_norm in ("azul", "shopping latam", "latam"):
            nome = f"🔵 {nome}"
        else:
            nome = nome

        list_nomes_clubes.append(nome)

    # --- Captura da pontuação ---
    list_pontuacao_clubes = []
    tabela_pm = dados_pagina.find('th', string=lambda s: s and 'Pontos e Milhas' in s)
    if tabela_pm:
        table = tabela_pm.find_parent('table')
        for tr in table.select('tbody tr'):
            nome_el = tr.select_one('td .d-grid span')
            if not nome_el:
                continue
            nome_norm = nome_el.get_text(strip=True).lower()

            if nome_norm not in nomes_alvo:
                continue

            tds = tr.find_all('td')
            if len(tds) < 2:
                continue

            texto_ganho = tds[-1].get_text(" ", strip=True)
            m = re.search(r'(\d+(?:[.,]\d+)?)\s*pt', texto_ganho, flags=re.I)
            if not m:
                continue

            valor = m.group(1).replace(',', '.')
            val_float = float(valor)
            val_final = int(val_float) if val_float.is_integer() else val_float

            list_pontuacao_clubes.append(val_final)

    # Monta DataFrame
    df = pd.DataFrame({
        "nome_clube": list_nomes_clubes,
        "pontuacao_clube": list_pontuacao_clubes
    })

    # --- Ajuste de pontuação e análise ---
    list_analise = []
    for index, row in df.iterrows():
        nome_norm = row["nome_clube"].lower()

        if ("livelo" not in nome_norm) and ("esfera" not in nome_norm):
            pontuacao_ajustada = row["pontuacao_clube"] / 1.8
        else:
            pontuacao_ajustada = row["pontuacao_clube"]

        #Analisando a promoção
        if pontuacao_ajustada >= 12:
          d_analise = '⭐⭐⭐⭐⭐ Nível 5 (Excelente / Raro) 😏: Promoção rara, aproveite sem medo!'
        elif pontuacao_ajustada >= 8:
          d_analise = '⭐⭐⭐⭐ Nível 4 (Muito Bom) 😎: Ótima Promoção, daqui pra cima já vale muito!'
        elif pontuacao_ajustada >= 6:
          d_analise = '⭐⭐⭐ Nível 3 (Bom) 😉: Bom momento para potencializar compras planejadas, mas se puder aguardar, tem coisa melhor!'
        elif pontuacao_ajustada >= 3:
          d_analise = '⭐⭐ Nível 2 (Mediano) 🧐: Dá pra usar se você realmente já iria comprar, mas é melhor aguardar algo melhor ;)'
        else:
          d_analise = '⭐ Nível 1 (Ruim) 😡: Sugiro aguardar algo melhor'

        list_analise.append(d_analise)

    df["analise_pontuacao"] = list_analise
    df["parceiro"] = parceiro
    df["data_coleta_promocao"] = data_formatada

    return df

### Controle de Envios (substitui as janelas de horário)

In [8]:
# =========================================
# 🔢 CONTROLE DE ENVIOS (substitui as janelas de horário)
# =========================================
# Em vez de disparar cada bloco com base na hora atual (hora_atual),
# cada execução consulta a planilha "Controle de Envios Grupo" para saber
# quantos envios já foram feitos HOJE e assume o próximo número da
# sequência (1º envio do dia, 2º envio do dia, etc).
#
# Isso resolve o problema do GitHub Actions atrasar e o cron cair fora
# da janela de horário esperada: se um gatilho atrasar ou for perdido,
# o próximo gatilho que rodar simplesmente assume o próximo número da
# sequência, em vez de pular aquele bloco para sempre naquele dia.
#
# IMPORTANTE: a planilha "Controle de Envios Grupo" é um arquivo separado
# da planilha de Feriados/Eventos/Base de Dados (SHEET_ID). É necessário:
# 1) Criar o secret SHEET_ID_CONTROLE_ENVIOS no GitHub Actions com o ID
#    dessa planilha (o trecho da URL entre /d/ e /edit).
# 2) Compartilhar a planilha com o e-mail da service account (campo
#    "client_email" dentro do secret GOOGLE_CREDENTIALS) como Editor.
# A primeira aba dela deve ter o cabeçalho: data | data_hora_execucao | numero_envio

import json
import gspread
from oauth2client.service_account import ServiceAccountCredentials

# 🐙 VERSÃO GITHUB ACTIONS (ativa)
SHEET_ID = os.environ["SHEET_ID"]
SHEET_ID_CONTROLE_ENVIOS = os.environ["SHEET_ID_CONTROLE_ENVIOS"]

# 🖥️ VERSÃO LOCAL (Colab)
#SHEET_ID = ler_config_local("sheet_id.txt")
#SHEET_ID_CONTROLE_ENVIOS = ler_config_local("sheet_id_controle_envios.txt")

# Ordem de envio do dia -> nome do bloco (usado só nos logs, não é gravado na planilha)
NOME_BLOCO_POR_NUMERO = {
    1: "Perfumaria e Cosméticos",
    2: "Roupas e Calçados Esportivos",
    3: "Moda",
    4: "Beleza e Cosméticos",
    5: "Varejo",
    6: "Transferências Bonificadas",
    7: "Jabares Noturnos",
}

def conectar_google_sheets():
    """Conecta na planilha de Feriados/Eventos/Base de Dados (SHEET_ID)"""
    scope = [
        "https://spreadsheets.google.com/feeds",
        "https://www.googleapis.com/auth/drive"
    ]

    # 🐙 VERSÃO GITHUB ACTIONS (ativa)
    creds_dict = json.loads(os.environ["GOOGLE_CREDENTIALS"])
    creds = ServiceAccountCredentials.from_json_keyfile_dict(creds_dict, scope)

    # 🖥️ VERSÃO LOCAL (Colab) — descomenta e comenta o bloco acima para testar local
    #from google.colab import files
    #uploaded = files.upload()  # faz upload do credenciais.json
    #creds = ServiceAccountCredentials.from_json_keyfile_name("credenciais.json", scope)

    client_gs = gspread.authorize(creds)
    spreadsheet = client_gs.open_by_key(SHEET_ID)
    return spreadsheet

def conectar_planilha_controle():
    """Conecta na planilha separada 'Controle de Envios Grupo' (SHEET_ID_CONTROLE_ENVIOS)
    e retorna a primeira aba dela, independente do nome que ela tenha."""
    scope = [
        "https://spreadsheets.google.com/feeds",
        "https://www.googleapis.com/auth/drive"
    ]

    creds_dict = json.loads(os.environ["GOOGLE_CREDENTIALS"])
    creds = ServiceAccountCredentials.from_json_keyfile_dict(creds_dict, scope)
    client_gs = gspread.authorize(creds)

    return client_gs.open_by_key(SHEET_ID_CONTROLE_ENVIOS).sheet1

def carregar_feriados(spreadsheet):
    """Carrega a aba Feriados e retorna um dict {date: nome_feriado}"""
    aba = spreadsheet.worksheet("Feriados")
    dados = aba.get_all_values()
    feriados = {}
    for row in dados[1:]:  # pula o cabeçalho
        if len(row) >= 2 and row[0] and row[1]:
            try:
                data = datetime.datetime.strptime(row[0].strip(), "%d/%m/%Y").date()
                feriados[data] = row[1].strip()
            except ValueError:
                continue
    return feriados

def carregar_eventos(spreadsheet):
    """Carrega a aba Eventos e retorna um dict {date: nome_evento}"""
    aba = spreadsheet.worksheet("Eventos")
    dados = aba.get_all_values()
    eventos = {}
    for row in dados[1:]:  # pula o cabeçalho
        if len(row) >= 2 and row[0] and row[1]:
            try:
                data = datetime.datetime.strptime(row[0].strip(), "%d/%m/%Y").date()
                eventos[data] = row[1].strip()
            except ValueError:
                continue
    return eventos

def gravar_no_sheets(spreadsheet, dados_row):
    """Adiciona uma nova linha na aba Base de Dados"""
    aba = spreadsheet.worksheet("Base de Dados")
    aba.append_row(dados_row, value_input_option="USER_ENTERED")
    print("✅ Linha gravada no Google Sheets!")

def carregar_envios_hoje(planilha_controle, data_formatada):
    """Lê a planilha Controle de Envios Grupo e retorna as linhas já registradas hoje"""
    dados = planilha_controle.get_all_values()
    return [row for row in dados[1:] if len(row) >= 1 and row[0].strip() == data_formatada]

def registrar_envio(planilha_controle, numero_envio):
    """Registra na planilha Controle de Envios Grupo que o envio N do dia foi disparado"""
    agora_str = agora_brasilia.strftime("%d/%m/%Y %H:%M:%S")
    linha = [data_formatada, agora_str, numero_envio]
    planilha_controle.append_row(linha, value_input_option="USER_ENTERED")
    print(f"📋 Envio {numero_envio} registrado na planilha às {agora_str}.")

# ── Conecta e descobre qual é o envio da vez ──
print("📊 Conectando ao Google Sheets...")
spreadsheet = conectar_google_sheets()
planilha_controle = conectar_planilha_controle()

envios_hoje = carregar_envios_hoje(planilha_controle, data_formatada)
numero_envio_hoje = len(envios_hoje) + 1

if numero_envio_hoje <= len(NOME_BLOCO_POR_NUMERO):
    nome_bloco_hoje = NOME_BLOCO_POR_NUMERO[numero_envio_hoje]
    print(f"➡️ Esse é o envio nº {numero_envio_hoje} de hoje: {nome_bloco_hoje}")
    # Registra a tentativa ANTES de rodar o bloco, para que uma falha no meio
    # do caminho não trave a sequência do dia inteiro (o próximo gatilho
    # já avança para o envio seguinte em vez de tentar o mesmo de novo).
    registrar_envio(planilha_controle, numero_envio_hoje)
else:
    nome_bloco_hoje = None
    print(f"✅ Todos os {len(NOME_BLOCO_POR_NUMERO)} envios de hoje já foram feitos. Nada a fazer nessa execução.")

📊 Conectando ao Google Sheets...
➡️ Esse é o envio nº 6 de hoje: Transferências Bonificadas
📋 Envio 6 registrado na planilha às 12/07/2026 11:39:09.


### Mensagem Bom dia

In [9]:
#Criando o texto de bom dia

dia_semana = agora_brasilia.strftime("%A")  # Nome do dia em inglês
hora_atual_saudacao = agora_brasilia.hour   # horário de Brasília

if dia_semana == "Monday":
    txt_dia_semana = "segundou! 😴"
elif dia_semana == "Tuesday":
    txt_dia_semana = "terçou!"
elif dia_semana == "Wednesday":
    txt_dia_semana = "quartou!"
elif dia_semana == "Thursday":
    txt_dia_semana = "quintou! 🚀"
elif dia_semana == "Friday":
    txt_dia_semana = "sextou! 🔥"
elif dia_semana == "Saturday":
    txt_dia_semana = "sabadou! 😎"
elif dia_semana == "Sunday":
    txt_dia_semana = "domingou! ☀️"
else:
    txt_dia_semana = "vocês estão bem?"

# Saudação dinâmica de acordo com o horário de Brasília. Antes a mensagem
# assumia que era sempre de manhã (fazia sentido quando esse envio era
# fixo na janela das 7h-8h). Agora que a ordem de envio é quem manda
# (numero_envio_hoje), esse bloco pode acabar rodando à tarde ou à noite
# se algum gatilho atrasar — então a saudação acompanha o horário real.
if 5 <= hora_atual_saudacao < 12:
    saudacao = "Bom dia"
    emoji_saudacao = "☀️"
    encerramento_saudacao = "Ótimo dia a todos!"
elif 12 <= hora_atual_saudacao < 18:
    saudacao = "Boa tarde"
    emoji_saudacao = "🌤️"
    encerramento_saudacao = "Ótima tarde a todos!"
else:
    saudacao = "Boa noite"
    emoji_saudacao = "🌙"
    encerramento_saudacao = "Ótima noite a todos!"

texto_bom_dia = f"""
{emoji_saudacao} *{saudacao} milheiros, {txt_dia_semana}*

Antes de começar, segue o nosso critério de análise para você aproveitar as melhores promoções no acúmulo de milhas:

Nós avaliamos cada promoção em 5 níveis, com base no retorno de milhas/pontos:

• *Nível 1 (Ruim) ⭐* Menos que 3 pontos → Sugiro aguardar algo melhor.
• *Nível 2 (Mediano) ⭐⭐* 3 pontos ou mais → Use só se você já iria comprar de qualquer forma.
• *Nível 3 (Bom) ⭐⭐⭐* 6 pontos ou mais → Bom para compras planejadas, mas ainda pode melhorar.
• *Nível 4 (Muito Bom) ⭐⭐⭐⭐* 8 pontos ou mais → Ótima promoção, daqui pra cima já vale muito a pena!
• *Nível 5 (Excelente / Raro) ⭐⭐⭐⭐⭐* 12 pontos ou mais → Promoção rara, dessas que não aparecem toda hora. Aproveite sem medo!

Quanto maior o nível, melhor o custo-benefício para acumular milhas/pontos.

Obs: Em promoções diretamente nas cias aéreas, nos aplicamos uma divisão de 1,80 para igualar as milhas ao pontos (considerando
uma transferência bonificada de 80%)!

Se você está gostando das nossas análises, deixe o seu like nessa mensagem ;)

{encerramento_saudacao} Bora pra cima! 🚀

Att @eumurilloborges
"""

## Grupo de Perfumaria (07h am)

### Loja Natura

In [10]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 1:
    df_natura = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-natura/",
        parceiro="Natura",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_natura)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Boticário

In [11]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 1:
    df_boticario = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-boticario/",
        parceiro="Boticário",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_boticario)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Eudora

In [12]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 1:
    df_eudora = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-eudora/",
        parceiro="Eudora",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_eudora)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Enviando os dados via Whatsapp

In [13]:
if numero_envio_hoje == 1:
  # Concatenando os dataframes
  df_final = pd.concat([df_natura, df_boticario, df_eudora], ignore_index=True)
  df_final

  # Criando a mensagem do grupo de Beleza e Cosméticos
  msg = ""
  msg_titulo = f"💄 *Resumo das Promoções da Categoria de Perfumaria e Cosméticos em {data_formatada}: 👇*"

  for index, row in df_final.iterrows():
    msg += textwrap.dedent(f"""
  *Parceiro:* {row["parceiro"]}
  *Programa de Fidelidade:* {row["nome_clube"]}
  *Pontuação:* {row["pontuacao_clube"]}
  *Análise da Promoção:* {row["analise_pontuacao"]}

  {"-"*30}""").strip()

  msg_grupo = msg_titulo + "\n""\n" + msg

  # Mensagem de Bom dia
  message_bom_dia = texto_bom_dia
  message_promo = msg_grupo

  # Payload
  payload = {
      "jid": id_grupo_envio,
      "message": message_bom_dia,
  }

  #Enviando a mensagem de bom dia
  response = requests.post(url, json=payload, headers=headers)

  # Resposta
  if response.status_code == 200:
      print("Mensagem de bom dia enviada com sucesso para o grupo!")
  else:
      print(f"Erro ao enviar mensagem: {response.status_code} - {response.text}")


  # Aguardar 2 minutos
  time.sleep(120)

  # Payload
  payload = {
      "jid": id_grupo_envio, #group_jid,
      "message": message_promo,
  }

  #Enviando as promoções
  response = requests.post(url, json=payload, headers=headers)

  # Resposta
  if response.status_code == 200:
      print("Mensagem com as promoções enviada com sucesso para o grupo!")
  else:
      print(f"Erro ao enviar mensagem: {response.status_code} - {response.text}")

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


##

## Grupo de Roupas e Calçados Esportivos (09h am) GRUPO OK

### Loja Centauro

In [14]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 2:
    df_centauro = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-centauro/",
        parceiro="Centauro",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_centauro)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja NetShoes

In [15]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 2:
    df_netshoes = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-netshoes/",
        parceiro="Netshoes",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_netshoes)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja New Balance

In [16]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 2:
    df_new_balance = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-new-balance/",
        parceiro="New Balance",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_new_balance)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Nike

In [17]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 2:
    df_nike = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-nike/",
        parceiro="Nike",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_nike)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Adidas

In [18]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 2:
    df_adidas = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-adidas/",
        parceiro="Adidas",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_adidas)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Enviando os dados via Whatsapp

In [19]:
if numero_envio_hoje == 2:
  # Concatenando os dataframes
  df_final = pd.concat([df_centauro, df_netshoes, df_new_balance, df_nike, df_adidas], ignore_index=True)
  df_final

  # Criando a mensagem do grupo de Beleza e Cosméticos
  msg = ""
  msg_titulo = f"👟🏓 *Promoções da Categoria de Roupas e Calçados Esportivos em {data_formatada}: 👇*"

  for index, row in df_final.iterrows():
    msg += textwrap.dedent(f"""
  *Parceiro:* {row["parceiro"]}
  *Programa de Fidelidade:* {row["nome_clube"]}
  *Pontuação:* {row["pontuacao_clube"]}
  *Análise da Promoção:* {row["analise_pontuacao"]}

  {"-"*30}""").strip()

  msg_grupo = msg_titulo + "\n\n" + msg

  # Mensagem
  message_promo = msg_grupo

  # Payload
  payload = {
      "jid": id_grupo_envio, #group_jid,
      "message": message_promo,
  }

  #Enviando as promoções
  response = requests.post(url, json=payload, headers=headers)

  # Resposta
  if response.status_code == 200:
      print("Mensagem com as promoções enviada com sucesso para o grupo!")
  else:
      print(f"Erro ao enviar mensagem: {response.status_code} - {response.text}")

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")


Não é o envio da vez hoje (ou este envio já foi feito).


## Grupo de Moda (11h am)

### Loja Renner

In [20]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 3:
    df_renner = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-lojas-renner/",
        parceiro="Lojas Renner",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_renner)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Riachuelo

In [21]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 3:
    df_riachuelo = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-riachuelo/",
        parceiro="Riachuelo",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_riachuelo)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja CEA

In [22]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 3:
    df_cea = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-cea/",
        parceiro="CEA",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_cea)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Hering

In [23]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 3:
    df_hering = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-hering/",
        parceiro="CEA",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_hering)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Dafiti

In [24]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 3:
    df_dafiti = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-dafiti/",
        parceiro="Dafiti",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_dafiti)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Enviando os dados via Whatsapp

In [25]:
if numero_envio_hoje == 3:

  # Concatenando os dataframes
  df_final = pd.concat([df_renner, df_riachuelo, df_cea, df_hering, df_dafiti], ignore_index=True)

  # Criando a mensagem do grupo de Beleza e Cosméticos
  msg = ""
  msg_titulo = f"👗👔 *Promoções da Categoria de Moda em {data_formatada}: 👇*"

  for index, row in df_final.iterrows():
    msg += textwrap.dedent(f"""
  *Parceiro:* {row["parceiro"]}
  *Programa de Fidelidade:* {row["nome_clube"]}
  *Pontuação:* {row["pontuacao_clube"]}
  *Análise da Promoção:* {row["analise_pontuacao"]}

  {"-"*30}""").strip()

  msg_grupo = msg_titulo + "\n\n" + msg

  # Mensagem
  message_promo = msg_grupo

  # Payload
  payload = {
      "jid": id_grupo_envio,
      "message": message_promo,
  }

  #Enviando as promoções
  response = requests.post(url, json=payload, headers=headers)

  # Resposta
  if response.status_code == 200:
      print("Mensagem com as promoções enviada com sucesso para o grupo!")
  else:
      print(f"Erro ao enviar mensagem: {response.status_code} - {response.text}")

  display(df_final)

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")


Não é o envio da vez hoje (ou este envio já foi feito).


## Grupo de Beleza e Cosméticos (13h am)

### Loja Beleza na Web

In [26]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 4:
    df_beleza_web = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-beleza-na-web/",
        parceiro="Beleza na Web",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_beleza_web)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Época Cosméticos

In [27]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 4:
    df_epoca_cosmeticos = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-epoca-cosmeticos/",
        parceiro="Época Cosméticos",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_epoca_cosmeticos)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Sephora

In [28]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 4:
    df_sephora = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-sephora/",
        parceiro="Sephora",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_sephora)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Oceane

In [29]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 4:
    df_oceane = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-oceane/",
        parceiro="Oceane",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_oceane)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Enviando os dados via Whatsapp

In [30]:
if numero_envio_hoje == 4:
  # Concatenando os dataframes
  df_final = pd.concat([df_beleza_web, df_epoca_cosmeticos, df_sephora, df_oceane], ignore_index=True)

  # Criando a mensagem do grupo de Beleza e Cosméticos
  msg = ""
  msg_titulo = f"💄 *Resumo das Promoções da Categoria de Beleza e Cosméticos em {data_formatada}: 👇*"

  for index, row in df_final.iterrows():
    msg += textwrap.dedent(f"""
  *Parceiro:* {row["parceiro"]}
  *Programa de Fidelidade:* {row["nome_clube"]}
  *Pontuação:* {row["pontuacao_clube"]}
  *Análise da Promoção:* {row["analise_pontuacao"]}

  {"-"*30}""").strip()

  msg_grupo = msg_titulo + "\n""\n" + msg

  # Mensagem
  message_promo = msg_grupo

  # Payload
  payload = {
      "jid": id_grupo_envio,
      "message": message_promo,
  }

  #Enviando as promoções
  response = requests.post(url, json=payload, headers=headers)

  # Resposta
  if response.status_code == 200:
      print("Mensagem com as promoções enviada com sucesso para o grupo!")
  else:
      print(f"Erro ao enviar mensagem: {response.status_code} - {response.text}")

  display(df_final)

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")


Não é o envio da vez hoje (ou este envio já foi feito).


## Grupo de Varejo (15h am)

### Loja Kabum

In [31]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 5:
    df_kabum = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-kabum/",
        parceiro="Kabum",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_kabum)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Ponto

In [32]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 5:
    df_ponto = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-ponto/",
        parceiro="Ponto",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_ponto)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Casas Bahia

In [33]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 5:
    df_casas_bahia = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-casas-bahia/",
        parceiro="Casas Bahia",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_casas_bahia)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Extra

In [34]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 5:
    df_extra = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-extra/",
        parceiro="Extra",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_extra)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Magalu

In [35]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 5:
    df_magalu = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-magazine-luiza/",
        parceiro="Magalu",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_magalu)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja Mercado Livre

In [36]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 5:
    df_mercado_livre = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-mercado-livre/",
        parceiro="Mercado Livre",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_mercado_livre)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Loja FastShop

In [37]:
hora_atual = agora_brasilia.hour

if numero_envio_hoje == 5:
    df_fast_shop = coletar_promocoes(
        url="https://www.comparemania.com.br/cashback-fast-shop/",
        parceiro="Fast Shop",
        data_formatada=data_formatada,
        nomes_alvo=nomes_alvo
    )
    display(df_fast_shop)
else:
    print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Enviando os dados via Whatsapp

In [38]:
if numero_envio_hoje == 5:

  # Concatenando os dataframes
  df_final = pd.concat([df_kabum, df_mercado_livre, df_magalu, df_ponto, df_casas_bahia, df_extra, df_fast_shop], ignore_index=True)
  df_final

  # Criando a mensagem do grupo de Beleza e Cosméticos
  msg = ""
  msg_titulo = f"📱💻 *Promoções do Grupo de Varejo em {data_formatada}: 👇*"

  for index, row in df_final.iterrows():
    msg += textwrap.dedent(f"""
  *Parceiro:* {row["parceiro"]}
  *Programa de Fidelidade:* {row["nome_clube"]}
  *Pontuação:* {row["pontuacao_clube"]}
  *Análise da Promoção:* {row["analise_pontuacao"]}

  {"-"*30}""").strip()

  msg_grupo = msg_titulo + "\n""\n" + msg

  # Mensagem
  message_promo = msg_grupo

  # Payload
  payload = {
      "jid": id_grupo_envio, #group_jid,
      "message": message_promo,
  }

  #Enviando as promoções
  response = requests.post(url, json=payload, headers=headers)

  # Resposta
  if response.status_code == 200:
      print("Mensagem com as promoções enviada com sucesso para o grupo!")
  else:
      print(f"Erro ao enviar mensagem: {response.status_code} - {response.text}")

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")


Não é o envio da vez hoje (ou este envio já foi feito).


## Jabares das 11h am (Mensagens com Imagens)

### Jabar banco inter (11h am Quarta e Sábado) GRUPO OK

In [39]:
if numero_envio_hoje == 3 and dia_semana in ("Wednesday", "Saturday"):

  # Aguardar 2 minutos
  time.sleep(120)

  if dia_semana == "Wednesday":
    message_promo = textwrap.dedent("""
    💳 *Quer um cartão de fácil acesso com pontuação e Salas VIPs?*

    O Inter Prime é uma ótima escolha: pontua bem, os pontos não expiram e você ainda
    ganha acesso às Salas VIP de forma gratuita, perfeito para dar um upgrade nas suas viagens.

    Lembrando que a forma mais fácil de conseguir esse cartão é *assinando o plano anual do Duo Gourmet!*

    Acesse o nosso site abaixo para garantir o cupom de desconto no Duo Gormet anual e também o bônus em pontos na abertura da sua conta Inter👇
    🔗 https://murilloborges.com.br/ranking-cartao-de-credito#inter-prime


    -----------------------------

    💳 Não conhece esse cartão? Assista o vídeo abaixo para conhecer os principais benefícios do cartão Inter Prime 👇
    https://www.youtube.com/watch?v=laEm4960klw&list=PLywCMnijM298ruCE-Cey18WvsoV2UqqQf&index=1""").strip()

  else:
    message_promo = textwrap.dedent("""
    *Pensando em começar no mundo dos cartões? 💳*

    O Inter Prime é uma excelente opção: boa pontuação, pontos que não expiram e acesso às Salas VIP ✈️🔥

    A maneira mais simples de conseguir esse cartão é assinando o plano Duo Gourmet anual 👇

    💰 *Quer economizar até R$50 no plano anual do Duo Gourmet?*
    Use o código *B1EE1054* no app do Inter
    👉 https://intergo.app/1f148152

    🟠 *Ainda não tem conta? abra sua conta no Banco Inter e ganhe 200 pontos Loop*
    Use o código *OW1AIMUA*
    👉 https://inter-co.onelink.me/Qyu7/ste2n6tb

    -----------------------------

    💳 Conheça os benefícios do cartão Inter Prime:
    https://www.youtube.com/watch?v=laEm4960klw&list=PLywCMnijM298ruCE-Cey18WvsoV2UqqQf&index=1""").strip()

  drive_id = "1OpOK2Ee1q2yJO-UDhfP3w3hFh2uM7y0s"
  img_url = f"https://drive.google.com/uc?export=view&id={drive_id}"

  payload = {
      "jid": id_grupo_envio, #group_jid,
      "mediaType": "image",
      "mimetype": "image/jpeg",
      "media": img_url,
      "caption": message_promo,  # ← usa seu texto de análise
      "filename": "analise.jpg"
  }

  response = requests.post(url_media, json=payload, headers=headers)

  print(response.json())

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Jabar planilha (11h am Quinta e Sexta) GRUPO OK

In [40]:
if numero_envio_hoje == 3 and dia_semana in ("Thursday", "Sunday"):

  # Aguardar 2 minutos
  time.sleep(120)

  if dia_semana == "Thursday":
    message_promo = textwrap.dedent("""
    *Quer organizar suas finanças de um jeito simples e inteligente?*

    A nossa Planilha de Controle Financeiro foi feita para quem quer parar de gastar no automático
    e finalmente entender para onde o dinheiro está indo, de forma clara, rápida e sem complicação.

    Por que essa planilha é a solução perfeita para você?
    • Simples de usar e totalmente automatizada
    • Visão completa do seu fluxo de caixa
    • Dashboard visual e intuitivo

    💡 *E o melhor?*
    • Ela custa menos do que uma pizza 🍕
    • Você paga *uma única vez* e fica com a planilha *para sempre*.
    • E você ainda ganha uma aula passo a passo. Mesmo sem saber Excel, você vai conseguir usar sem dificuldade. 🎓✅
    • Ao adquirir, você ainda apoia o nosso trabalho e nos ajuda a continuar trazendo conteúdo gratuito aqui no grupo. 🙏

    👉 Assista ao vídeo e adquira sua planilha aqui:
    https://bit.ly/whatsplancontrolefinanceiro

    -----------------------------

    🧠 *Organização financeira muda tudo.*
    Comece hoje, seu “eu do futuro” vai agradecer.""").strip()
  else:
    message_promo = textwrap.dedent("""
    *Quer organizar suas finanças de um jeito simples?*

    Nossa Planilha de Controle Financeiro te ajuda a ver para onde seu dinheiro está indo e a controlar seus gastos sem complicação.

    • Custa menos que uma pizza 🍕
    • Pagou uma vez, é sua para sempre.
    • E tem aula passo a passo (mesmo se você não souber Excel). 🎓✅

    👉 Conheça e adquira aqui:
    https://bit.ly/whatsplancontrolefinanceiro""").strip()


  # Imagem do Google Drive
  drive_id = "1dHHTOXW8g3bZ4AGZk14Lvnu9mGSpHoVc"
  img_url = f"https://drive.google.com/uc?export=view&id={drive_id}"

  payload = {
      "jid": id_grupo_envio, #group_jid,
      "mediaType": "image",
      "mimetype": "image/jpeg",
      "media": img_url,
      "caption": message_promo,  # ← usa seu texto de análise
      "filename": "analise.jpg"
  }

  response = requests.post(url_media, json=payload, headers=headers)

  print(response.json())

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Jabar TopCashback (11h am Segunda e Quinta)

In [41]:
if numero_envio_hoje == 3 and dia_semana in ("Monday", "Thursday"):

  #Aguardar 60 segundos
  time.sleep(60)

  if dia_semana == "Monday":

    #Imagem do Google Drive
    drive_id = "1sVCoiFDKu0gbioPQSpZbOVkGbq6NYb8X" #Imagem de Segunda
    img_url = f"https://drive.google.com/uc?export=view&id={drive_id}"

  else:

    #Imagem do Google Drive
    drive_id = "1idhL58jQID5wemKh0XoHfqROn7L3MzTK" #Imagem de Quinta
    img_url = f"https://drive.google.com/uc?export=view&id={drive_id}"

  message_promo = textwrap.dedent("""
  🇬🇧 *Vai viajar ou reservar serviços no exterior? Então presta atenção nisso 👇*

  Alguns parceiros como o ALL por exemplo, permitem reservas no Brasil 🇧🇷

  Se você ainda não tem conta no *TopCashBack do Reino Unido*, vale muito a pena abrir.

  Usando o nosso link, você ainda pode ganhar um *bônus após a primeira compra* 💷✨

  👉 Abra sua conta aqui:
  https://bit.ly/topcashbackreinounido

  -----------------------------

  🏨 *Hospedagens com Cashback em Libra Esterlina:*
  • ALL Accor Live Limitless – até *15% de cashback*
    https://www.topcashback.co.uk/all-accor-live-limitless/

  • Hotels_com – até *15% de cashback*
    https://www.topcashback.co.uk/hotelscom/

  • Expedia – até *8,5% de cashback*
    https://www.topcashback.co.uk/search/merchants/?s=Expedia

  • Agoda – até *4% de cashback*
    https://www.topcashback.co.uk/search/merchants/?s=Agoda

  • Lastminute – até *8,5% de cashback*
    https://www.topcashback.co.uk/search/merchants/?s=lastminute.com

  -----------------------------

  🚗 *Locadoras de Carros com Cashback em Libra Esterlina:*
  • Avis – até *9,35% de cashback*
    https://www.topcashback.co.uk/search/merchants/?s=Avis

  • Discover Cars – até *25,5% de cashback*
    https://www.topcashback.co.uk/discover-car-hire/

  • Europcar – até *11,47% de cashback*
    https://www.topcashback.co.uk/europcar/

  • Sixt UK – até *8,5% de cashback*
    https://www.topcashback.co.uk/sixt-uk/

  • Alamo – até *4,25% de cashback*
    https://www.topcashback.co.uk/alamo/

  -----------------------------

  🚆 *Trem, Ferry e Ônibus com Cashback em Libra Esterlina:*
  • Omio – até *4,25% de cashback*
    https://www.topcashback.co.uk/omio/

  -----------------------------

  🎟️ *Passeios e Atrações com Cashback em Libra Esterlina:*
  • GetYourGuide – até *11% de cashback*
    https://www.topcashback.co.uk/getyourguide/

  • Groupon – até *5,95% de cashback*
    https://www.topcashback.co.uk/groupon/

  • Tripadvisor – até *12% de cashback*
    https://www.topcashback.co.uk/tripadvisor-tours-and-experiences/

  • Disneyland Paris – até *£42.50 de cashback*
    https://www.topcashback.co.uk/disneyland-resort-paris/

  -----------------------------

  ⚠️ *Importante:* sempre leia os termos do cashback antes de comprar.

  ‼️ O percentual pode variar diariamente e cada parceiro possui regras específicas para a validação do retorno.
    """).strip()


  message_promo2 = textwrap.dedent("""
  🇺🇸 *Além do TopCashBack Britânico, também existe o TopCashBack Americano 👇*

  Por isso, a melhor estratégia é sempre consultar os dois sites e comparar qual deles está oferecendo o *maior cashback no momento*.

  Em muitos casos, a diferença pode ser bem relevante 💰

  Alguns parceiros permitem reservas mesmo estando no Brasil 🇧🇷, o que pode gerar cashback em dólar de forma bem interessante 💵

  Se você ainda não tem conta no *TopCashBack dos Estados Unidos*, vale muito a pena abrir.
  Usando o nosso link, você ainda pode ganhar um *bônus após a primeira compra* ✨

  👉 Abra sua conta aqui:
  https://bit.ly/topcashbackamericano

  -----------------------------

  🏨 *Hospedagens*
  • ALL Accor – até *15% de cashback*
    https://www.topcashback.com/accorhotels/

  • Booking – até *10% de cashback*
    https://www.topcashback.com/booking-com/

  • Hotels.com – até *15% de cashback*
    https://www.topcashback.com/hotels-com/

  • Expedia – até *8,5% de cashback*
    https://www.topcashback.com/expedia/

  • Agoda – até *4% de cashback*
    https://www.topcashback.com/agoda/

  -----------------------------

  🚗 *Locadoras de Carros*
  • Avis – até *9,35% de cashback*
    https://www.topcashback.com/avis-rent-a-car/

  • Hertz – até *5% de cashback*
    https://www.topcashback.com/hertz/

  • Europcar – até *9% de cashback*
    https://www.topcashback.com/europcar/

  • Sixt – até *8% de cashback*
    https://www.topcashback.com/sixt/

  • Alamo – até *3% de cashback*
    https://www.topcashback.com/alamo-rent-a-car/

  -----------------------------

  🚆 *Trem, Ferry e Ônibus*
  • Omio – até *10% de cashback*
    https://www.topcashback.com/omio/

  -----------------------------

  🎟️ *Passeios e Atrações*
  • GetYourGuide – até *7% de cashback*
    https://www.topcashback.com/getyourguide/

  • Groupon – até *4% de cashback*
    https://www.topcashback.com/groupon/

  • Viator – até *15% de cashback*
    https://www.topcashback.com/viator/

  • Tripadvisor – até *10% de cashback*
    https://www.topcashback.com/tripadvisor-hotels/

  -----------------------------

  ⚠️ *Importante:* sempre leia os termos do cashback antes de comprar.

  ‼️ O percentual pode variar diariamente e cada parceiro possui regras específicas para a validação do retorno.
  """).strip()


  #Mensagem 1
  payload = {
      "jid": id_grupo_envio,
      "mediaType": "image",
      "mimetype": "image/jpeg",
      "media": img_url,
      "caption": message_promo,
      "filename": "analise.jpg"
  }

  headers = {
      "X-Api-Key": API_KEY,
      "Content-Type": "application/json"
  }

  response = requests.post(url_media, json=payload, headers=headers)

  #Aguardar 20 segundos
  time.sleep(20)


  #Mensagem 2
  message_promo = message_promo2

  payload = {
      "jid": id_grupo_envio,
      "message": message_promo,
  }

  response = requests.post(url, json=payload, headers=headers)

  print("Mensagem enviada!")

  #Aguardar 20 segundos
  time.sleep(20)


  #Mensagem 3
  message_promo3 = textwrap.dedent("""
    📺 *Quer aprender a usar o TopCashBack do jeito certo e extrair o máximo de cashback?*

    No nosso canal do YouTube temos uma *playlist completa* dedicada ao TopCashBack, onde mostramos passo a passo como usar a plataforma, dicas práticas, cuidados importantes e estratégias para aumentar seus ganhos 💰

    É conteúdo ideal tanto para quem está começando quanto para quem já usa e quer melhorar os resultados.

    👉 Assista à playlist completa aqui:
    https://www.youtube.com/playlist?list=PLywCMnijM298Xa4SBJ7w4aQK_qsxgG5nJ
  """).strip()

  message_promo = message_promo3

  payload = {
      "jid": id_grupo_envio,
      "message": message_promo,
  }

  response = requests.post(url, json=payload, headers=headers)

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")



Não é o envio da vez hoje (ou este envio já foi feito).


## Jabares das 17h am (Mensagens com Imagens)

### Jabar Visto Americano Hit The Change (5 envio na Segunda) GRUPO OK

In [42]:
from datetime import datetime
from zoneinfo import ZoneInfo

# Data atual em Brasília
agora = datetime.now(ZoneInfo("America/Sao_Paulo"))

numero_semana = agora.isocalendar().week

if numero_envio_hoje == 5 and dia_semana == "Monday":

    # Aguardar 1 minuto
    time.sleep(60)

    if numero_semana % 2 == 0:
        print("Semana par")

        drive_id = "1WBD7goFmjB1b_ag3j-uGTaa0sLc0_tn-"

        message_promo = textwrap.dedent("""
        🇺🇸 *Vai tirar o visto americano pela primeira vez ou renovar? Não deixe essa etapa ao acaso!*

        Somos parceiros da Hit The Change, uma das principais assessorias especializadas em visto americano, e conseguimos um benefício exclusivo para vocês.

        💰 Pela nossa indicação você ganha R$ 50 de desconto na assessoria para o visto americano.

        Solicitar um visto envolve um investimento importante e estar bem orientado durante todo o processo pode fazer a diferença. Conte com uma equipe especializada para acompanhar você desde a documentação até a entrevista consular.

        📲 Clique no link abaixo para falar com a equipe e garantir seu desconto exclusivo 👇
        https://bit.ly/vistoamericanohtc
        """).strip()

    else:
        print("Semana ímpar")

        drive_id = "1nWonSndcpBmzCmfUmFKrGSBJu83SiHuW"

        message_promo = textwrap.dedent("""
        ✈️ *Seu sonho de conhecer os Estados Unidos começa pelo visto americano*

        Somos parceiros da Hit The Change, uma assessoria especializada que auxilia brasileiros em todo o processo de solicitação do visto americano e renovação. 🇺🇸

        Além de contar com o suporte de uma equipe experiente, quem chega pela nossa indicação garante R$ 50 de desconto na assessoria.

        📲 Clique no link abaixo para conversar com a equipe e aproveitar esse benefício exclusivo.
        https://bit.ly/vistoamericanohtc
        """).strip()

    # Código comum
    img_url = f"https://drive.google.com/uc?export=view&id={drive_id}"

    payload = {
        "jid": id_grupo_envio,
        "mediaType": "image",
        "mimetype": "image/jpeg",
        "media": img_url,
        "caption": message_promo,
        "filename": "analise.jpg"
    }

    headers = {
        "X-Api-Key": API_KEY,
        "Content-Type": "application/json"
    }

    response = requests.post(url_media, json=payload, headers=headers)

    print("Mensagem enviada!")

## Jabares das 19h pm (Mensagens com Imagens)

### Jabar cartao Ruby Stell (19h Terça-Feira)

In [43]:
if numero_envio_hoje == 7 and dia_semana in ("Tuesday"):

  # Aguardar 1 minuto
  time.sleep(60)

  if dia_semana == "Friday":

    # Imagem do Google Drive
    drive_id = "1EV2mu8AfLkIzo9dM23nP58e5lN7bYN18" #Imagem de Sexta
    img_url = f"https://drive.google.com/uc?export=view&id={drive_id}"

    message_promo = textwrap.dedent("""
    *Vai viajar para fora do Brasil ou fazer uma compra internacional? Então presta atenção nisso 👇*

    O *Cartão Crypto Ruby* é simplesmente um dos melhores cartões para uso internacional, porque:
    • Não cobra IOF nas compras (3,5% de economia) 🌍
    • Tem spread super baixo 💱
    • *2% de cashback* em todas as compras 💸
    • Spotify grátis 🎧

    Ou seja: você gasta menos, economiza nas taxas e ainda recebe dinheiro de volta.

    -----------------------------

    💳 *Quer ver esse cartão funcionando na prática?*
    Eu testei esse cartão por 15 dias e gravei um vídeo mostrando como usar, benefícios e pontos de atenção 👇
    https://www.youtube.com/watch?v=OA7LlqiobHQ&t=300s

    -----------------------------

    🔵 Gostou do cartão?
    Abra sua conta na Crypto com o nosso link e ganhe até **$60 USD** de bônus 💵
    👉 https://bit.ly/wcryptoruby

    """).strip()

  else:

    # Imagem do Google Drive
    drive_id = "1eLpOE5cJJe2ahFIMbbY1do2RSE-adggr" #Imagem de Terça
    img_url = f"https://drive.google.com/uc?export=view&id={drive_id}"

    message_promo = textwrap.dedent("""
    *Se você vai viajar ou costuma comprar fora do Brasil, esse cartão pode te fazer economizar MUITO* ✈️💸

    O *Crypto Ruby* é um dos poucos cartões que realmente vale a pena no uso internacional:

    • Zero IOF nas compras (economia 3,5%) 🌍
    • Spread super baixo (conversão mais justa) 💱
    • *2% de cashback* em tudo que você gastar 💸
    • Spotify grátis 🎧

    👀 *Spoiler:* ele tem um custo muito menor que vários cartões globais famosos do mercado.

    -----------------------------

    🎥 *Quer ver como ele funciona na prática?*
    Eu usei esse cartão e gravei um vídeo mostrando tudo:
    https://www.youtube.com/watch?v=OA7LlqiobHQ&t=300s

    -----------------------------

    🔵 Curtiu a ideia?
    Abra sua conta pela Crypto e ganhe até *$60 USD* de bônus 💵
    👉 https://bit.ly/wcryptoruby
    """).strip()

  payload = {
      "jid": id_grupo_envio, #group_jid,
      "mediaType": "image",
      "mimetype": "image/jpeg",
      "media": img_url,
      "caption": message_promo,
      "filename": "analise.jpg"
  }

  headers = {
      "X-Api-Key": API_KEY,
      "Content-Type": "application/json"
  }

  response = requests.post(url_media, json=payload, headers=headers)

  print("Mensagem enviada!")

else:
  print("Não é o envio da vez hoje (ou este envio já foi feito).")

Não é o envio da vez hoje (ou este envio já foi feito).


### Jabar Passeios Get Your Guide (19h Terça-Feira e sexta-Feira)

In [44]:
if numero_envio_hoje == 7 and dia_semana in ("Tuesday", "Friday"):

    # Aguardar 5 segundos
    time.sleep(5)

    drive_id = "1F3Hq6nsn4fzS5KwZy4Addf65sY1AW7AG"
    img_url = f"https://drive.google.com/uc?export=view&id={drive_id}"

    if dia_semana == "Monday":
        message_promo = textwrap.dedent("""
        🌍 *Quer transformar sua viagem em uma experiência inesquecível?*

        No Get Your Guide você encontra os melhores passeios, tours e experiências em destinos do mundo todo — tudo em um só lugar! 🗺️

        🎯 City tours, passeios de barco, experiências gastronômicas, trilhas, museus e muito mais, em centenas de destinos pelo mundo.

        👉 *Acesse nossas recomendações e escolha o passeio perfeito para a sua próxima viagem:*
        https://www.getyourguide.com/explorer/creators/-cs4294

        💰 *Primeira reserva? Use o cupom abaixo e ganhe 5% de desconto:*
        🎟️ Cupom: *MURILLOINVESTOR5*

        -----------------------------

        ✈️ Viagem planejada é viagem aproveitada. Garante já o seu passeio!
        """).strip()

    else:  # Thursday
        message_promo = textwrap.dedent("""
        ✈️ *Já tem viagem marcada? Então esse conteúdo é pra ti!*

        No Get Your Guide tu encontras passeios incríveis em destinos do mundo todo — desde city tours clássicos até experiências únicas que vão fazer a tua viagem ser ainda mais especial 🌍🔥

        🗺️ Temos opções separadas por país pra facilitar a tua busca!

        👉 *Dá uma olhada nas nossas recomendações:*
        https://www.getyourguide.com/explorer/creators/-cs4294

        💰 *Na tua primeira reserva, usa o cupom abaixo e garante 5% de desconto:*
        🎟️ *MURILLOINVESTOR5*

        -----------------------------

        🌟 Experiências boas não acontecem por acaso — elas são planejadas!
        """).strip()

    payload = {
        "jid": id_grupo_envio,
        "mediaType": "image",
        "mimetype": "image/jpeg",
        "media": img_url,
        "caption": message_promo,
        "filename": "getyourguide.jpg"
    }

    response = requests.post(url_media, json=payload, headers=headers)
    print("Mensagem enviada!")

else:
    print("Não é o envio da vez hoje, ou hoje não é dia de Get Your Guide.")

Não é o envio da vez hoje, ou hoje não é dia de Get Your Guide.


## Transferências Bonificadas

In [45]:
if numero_envio_hoje == 6:

  import requests
  from bs4 import BeautifulSoup
  from google import genai
  from google.genai import types
  import re
  import datetime
  from zoneinfo import ZoneInfo
  import os

  # =========================================
  # 🔑 CHAVE DA API DA IA (Gemini)
  # =========================================
  client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

  # =========================================
  # 🔐 GOOGLE SHEETS
  # =========================================
  # A conexão (spreadsheet), feriados e eventos já foram carregados uma
  # única vez lá no topo do notebook, na célula "Controle de Envios".
  # Aqui só recarregamos feriados/eventos para garantir dados atualizados.
  feriados = carregar_feriados(spreadsheet)
  eventos = carregar_eventos(spreadsheet)
  print(f"✅ {len(feriados)} feriados e {len(eventos)} eventos carregados.")

  # =========================================
  # 🔎 EXTRAÇÃO DE LINKS DE PROMOÇÕES
  # =========================================
  def extrair_links_promocoes():

      # 🐙 VERSÃO GITHUB ACTIONS (ativa)
      url = os.environ["URL_SCRAPING"]
      url_two = os.environ["URL_SCRAPING_TWO"]

      # 🖥️ VERSÃO LOCAL (Colab)
      #url = ler_config_local("url_scraping.txt")
      #url_two = ler_config_local("url_scraping_two.txt")

      headers = {"User-Agent": "Mozilla/5.0"}
      response = requests.get(url, headers=headers)
      soup = BeautifulSoup(response.text, "html.parser")

      links = []
      for a in soup.find_all("a", href=True):
          href = a["href"]
          if (
              url_two in href
              and "pontos" in href
              and not any(x in href for x in ["categoria", "tag", "page", "#"])
          ):
              links.append(href)

      # dict.fromkeys() remove duplicatas preservando a ORDEM de inserção
      # (diferente de set(), que embaralha). O site lista as matérias mais
      # recentes primeiro no HTML, então isso garante processar/enviar da
      # mais nova pra mais antiga, em vez de ordem aleatória por hash.
      return list(dict.fromkeys(links))

  def carregar_status_links(spreadsheet):
      """Lê a aba 'Status Links' e retorna um dict {link: status}, sempre
      com o registro mais recente de cada link. Usado para pular só os
      links já CONFIRMADOS como vencidos — os demais (novos ou ainda
      válidos) continuam sendo reprocessados, pra permitir reenviar como
      lembrete enquanto a promoção durar."""
      aba = spreadsheet.worksheet("Status Links")
      dados = aba.get_all_values()
      status_por_link = {}
      for row in dados[1:]:  # pula o cabeçalho
          if len(row) >= 2 and row[0]:
              link = row[0].strip()
              status = row[1].strip()
              status_por_link[link] = status  # linhas mais abaixo sobrescrevem (mais recentes)
      return status_por_link

  def registrar_status_link(spreadsheet, link, status, data_validade):
      """Registra o status mais recente de um link (Dentro do prazo / Fora
      do prazo) na aba 'Status Links' — usado nas próximas execuções pra
      decidir se ele deve ser pulado (só quando vencido) ou reprocessado."""
      aba = spreadsheet.worksheet("Status Links")
      agora_str = agora_brasilia.strftime("%d/%m/%Y %H:%M:%S")
      linha = [link, status, data_validade or "", agora_str]
      aba.append_row(linha, value_input_option="USER_ENTERED")

  # =========================================
  # 🤖 EXTRAÇÃO COM IA
  # =========================================
  def extrair_com_ia(html_texto, data_publicacao):
    from datetime import datetime
    from zoneinfo import ZoneInfo
    hoje = datetime.now(ZoneInfo("America/Sao_Paulo")).strftime("%d/%m/%Y")

    prompt = f"""
    Você é um especialista em milhas aéreas e vai extrair dados importantes e montar uma mensagem que será enviada em um grupo.
    Seja preciso e nunca invente informações que não estejam explícitas no texto.

    ─────────────────────────────────────
    PARCEIRO (banco de origem)
    ─────────────────────────────────────
    - O parceiro é sempre o DE (origem dos pontos) e a companhia aérea/programa é o PARA (destino das milhas).
      Exemplo: ESFERA → GOL (Smiles)
    - Se a promoção tiver apenas 1 parceiro identificável, use o nome dele conforme as regras abaixo.
    - Se tiver exatamente 2 parceiros, cite os dois nomes. Exemplo: ESFERA e Livelo
    - Se tiver 3 ou mais parceiros, use "Vários parceiros".
    - Nunca invente um parceiro se não estiver explícito no texto.
    - Nunca confunda banco com companhia aérea.

    ─────────────────────────────────────
    NOMES PADRONIZADOS
    ─────────────────────────────────────
    - Azul / Tudo Azul → AZUL (Tudo Azul)
    - GOL / Smiles → GOL (Smiles)
    - Latam / Latam Pass → Latam (Latam Pass)
    - Iberia → Iberia Club
    - Livelo → Livelo
    - Qualquer outro → primeira letra maiúscula, restante minúsculo

    ─────────────────────────────────────
    ÍCONES POR COMPANHIA/PROGRAMA DE DESTINO
    ─────────────────────────────────────
    - AZUL (Tudo Azul) → 🔵
    - Latam (Latam Pass) → 🔵
    - ALL Accor → 🔵
    - KrisFlyer → 🔵
    - GOL (Smiles) → 🟠
    - Iberia Club → 🔴
    - Sicredi → 🟢
    - Banco do Brasil / BB → 🟡
    - C6 Bank → ⚫️
    - Qualquer outro → ⚪️

    ─────────────────────────────────────
    DATA DE VALIDADE
    ─────────────────────────────────────
    - Esta matéria foi publicada em {data_publicacao}. Se o texto mencionar a
      validade citando SOMENTE o dia (sem mês explícito), use o mês e o ano
      da PUBLICAÇÃO da matéria acima — NUNCA use o mês/ano de hoje pra isso,
      porque esta matéria pode estar sendo reprocessada dias ou semanas
      depois de ter sido publicada.
    - A data de hoje é {hoje}. Use isso só como contexto geral, nunca para
      inferir o mês/ano de uma data escrita parcialmente no texto.
    - A promoção é válida até às 23h59 do dia informado.
    - Não confunda a data de publicação do artigo com a data de validade da promoção.
      A data de validade geralmente aparece como: "promoção se encerra em", "válida até", "transferências até".
    - Se não encontrar nenhuma data de validade no texto, escreva: 📅 Data não informada

    ─────────────────────────────────────
    COMPANHIA AÉREA vs PROGRAMA DE HOTEL
    ─────────────────────────────────────
    - Mostre ✈️ Companhia aérea: somente quando o destino for uma cia aérea.
    - Mostre 🏨 Programa de fidelidade de hotel: somente quando o destino for ALL Accor.
    - NUNCA mostre as duas linhas na mesma mensagem — é sempre uma ou outra.
    - O parceiro NUNCA é Azul/Tudo Azul quando o destino é ALL Accor. Sempre o contrário: parceiro ALL Accor → destino AZUL (Tudo Azul).

    ─────────────────────────────────────
    EXEMPLO PRÁTICO
    ─────────────────────────────────────
    - Padrão fixo: Enviou 10.000 pontos.
    - Calcule o Recebeu com base no bônus máximo informado.
      Exemplo: 100% de bônus → Enviou 10.000 pontos → Recebeu 20.000 milhas.
    - Use ponto como separador de milhar (padrão brasileiro): 20.000, não 20,000.
    - Sempre indique o percentual considerado: "Considerando X% de Bônus".

    ─────────────────────────────────────
    ALERTA FINAL
    ─────────────────────────────────────
    - Sempre inclua o aviso para ler o regulamento antes de participar.

    ─────────────────────────────────────
    TEXTO DA PROMOÇÃO PARA ANALISAR:
    ─────────────────────────────────────
    {html_texto}

    ─────────────────────────────────────
    FORMATO DE SAÍDA (siga exatamente):
    ─────────────────────────────────────

    🔵 Alerta de Transferência Bonificada: C6 BANK para AZUL (Tudo Azul) 🔵
    ━━━━━━━━━━

    🚀 Transfira seus pontos do C6 BANK para o programa Tudo Azul e ganhe milhas bônus!

    🏦 Parceiro: C6 BANK
    ✈️ Companhia aérea: AZUL (Tudo Azul)  [somente quando for cia aérea]
    🏨 Programa de fidelidade de hotel: ALL Accor  [somente quando o destino for ALL Accor]

    🔥 BONIFICAÇÃO: Até 130% de bônus (Consulte as regras para o seu percentual específico)

    📱 EXEMPLO PRÁTICO:
    Enviou: 10.000 pontos C6 BANK
    Recebeu: 23.000 milhas na Azul (Considerando 130% de Bônus)

    📅 Válido até 19/03/2026

    ⚠️ Atenção: Leia o regulamento da promoção antes de participar!
    """

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction="Extraia dados de promoções de milhas.",
            temperature=0,
        ),
    )

    return response.text

  # =========================================
  # 🗓️ FUNÇÕES AUXILIARES DE DATA
  # =========================================
  def nome_dia_semana_pt(data):
      dias = {0: "Segunda-feira", 1: "Terça-feira", 2: "Quarta-feira", 3: "Quinta-feira",
              4: "Sexta-feira", 5: "Sábado", 6: "Domingo"}
      return dias[data.weekday()]

  def nome_mes_pt(data):
      meses = {1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
              5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
              9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"}
      return meses[data.month]

  # =========================================
  # ▶️ EXECUÇÃO PRINCIPAL
  # =========================================
  list_promocao_key = []
  #print(f'Imprimindo a lista na primeira execucao: {list_promocao_key}')

  if __name__ == "__main__":

      links = extrair_links_promocoes()

      # ── Só pula links já CONFIRMADOS como vencidos. Os demais (novos ou
      # ainda válidos numa checagem anterior) continuam sendo reprocessados,
      # pra permitir reenviar a promoção como lembrete enquanto durar. ──
      status_links = carregar_status_links(spreadsheet)
      links_nao_vencidos = [l for l in links if status_links.get(l) != "Fora do prazo"]

      # ── Limita quantas matérias processar por execução ──
      # A lista já vem da mais recente pra mais antiga, então isso sempre
      # prioriza as promoções mais novas primeiro.
      MAX_LINKS_POR_EXECUCAO = 15
      links_para_processar = links_nao_vencidos[:MAX_LINKS_POR_EXECUCAO]

      print(f"📋 {len(links)} links encontrados | {len(links_nao_vencidos)} não vencidos | processando até {len(links_para_processar)} nesta execução.")

      for i, link in enumerate(links_para_processar, start=1):
          print(f"\n{'='*50}")
          #print(f"{i}. {link}")

          headers = {"User-Agent": "Mozilla/5.0"}
          response = requests.get(link, headers=headers)
          soup = BeautifulSoup(response.text, "html.parser")
          html_texto = soup.get_text(separator=" ", strip=True)

          # ── Data real de publicação da matéria (extraída do HTML, não da IA) ──
          meta_publicado = soup.find("meta", attrs={"property": "article:published_time"})
          if meta_publicado and meta_publicado.get("content"):
              data_publicacao_dt = datetime.datetime.fromisoformat(meta_publicado["content"]).astimezone(ZoneInfo("America/Sao_Paulo"))
              data_publicacao_str = data_publicacao_dt.strftime("%d/%m/%Y")
          else:
              data_publicacao_dt = None
              data_publicacao_str = "Data de publicação não encontrada"

          print(f"Publicada em: {data_publicacao_str}")

          resultado = extrair_com_ia(html_texto, data_publicacao_str)
          msg_link_grupo = f"🔗 Mais informações sobre a promoção: {link}"
          resultado += f"\n\n{msg_link_grupo}\n\n"

          # ── Extrai data de validade ──
          match = re.search(r"\d{2}/\d{2}/\d{4}", resultado)

          if match:
              data_str = match.group()
              data_formatada = datetime.datetime.strptime(data_str, "%d/%m/%Y")
              data_promocao = data_formatada.date()
              data_atual = datetime.datetime.now(ZoneInfo("America/Sao_Paulo")).date()

              # ── Trava de segurança: validade não pode ser antes da publicação ──
              if data_publicacao_dt and data_promocao < data_publicacao_dt.date():
                  status_promo = "Fora do prazo"
                  print(f"⚠️ Data de validade ({data_str}) é anterior à publicação ({data_publicacao_str}) — descartando.")
              else:
                  status_promo = "Dentro do prazo" if data_promocao >= data_atual else "Fora do prazo"
          else:
              data_str = None
              data_promocao = None
              status_promo = "Fora do prazo"

          print(f"Status: {status_promo}")

          # ── Registra o status atual do link, pra decidir nas próximas
          # execuções se ele deve ser pulado (só quando vencido) ou não ──
          registrar_status_link(spreadsheet, link, status_promo, data_str)

          # ── Alerta de último dia — lógica no Python, não na IA ──
          if data_promocao and data_promocao == data_atual:
            alerta_hoje = "\n\n🚨 ATENÇÃO: Essa promoção acaba HOJE! Corra para aproveitar!\n"
            resultado = re.sub(
                r"(📅 Váli?do? até \d{2}/\d{2}/\d{4})",
                r"\1" + alerta_hoje,
                resultado
            )

          # ── Extrai campos da mensagem ──
          parceiro = re.search(r"🏦 Parceiros?:\s*(.+)", resultado)
          parceiro = parceiro.group(1).strip() if parceiro else None

          cia = re.search(r"✈️ Companhia aérea:\s*(.+)", resultado)
          cia = cia.group(1).strip() if cia else None

          if not cia:
              hotel = re.search(r"🏨 Programa de fidelidade de hotel:\s*(.+)", resultado)
              cia = hotel.group(1).strip() if hotel else None

          bonus = re.search(r"BONIFICAÇÃO:.*?(\d+)%", resultado)
          bonus_str = bonus.group(1) + "%" if bonus else None
          bonus_num = int(bonus.group(1)) if bonus else None

          # ── Link Clube Azul — lógica no Python, não na IA ──
          LINK_CLUBE_AZUL = "https://apps.voeazul.com.br/TudoAzulClub/share-mgm.html?hash=7uhPhMsj%2B2YY%2Bczpr2ebrxpa6feTjo%2F5yu2MJlXj4ysc-aC"
          TEXTO_CLUBE_AZUL = f"\n💰 Assine o Clube Azul com o nosso link para ganhar a bonificação maior + 1.000 milhas bônus (a gente também ganha esse bônus): {LINK_CLUBE_AZUL}\n"

          LINK_CLUBE_SMILES = "https://www.smiles.com.br/indicar-amigos/indicado?icode=1-123690609266"
          CODIGO_INDICACAO_SMILES = '1-123690609266'
          TEXTO_CLUBE_SMILES = f"\n💰 Assine o Clube Smiles com o nosso link para ganhar a bonificação maior + 1.000 milhas bônus (a gente também ganha esse bônus): {LINK_CLUBE_SMILES}\n Obs: Se o código não for aplicado automaticamente, adicione o código {CODIGO_INDICACAO_SMILES} na adesão do clube 🚀\n\n"

          if cia and "azul" in cia.lower():
              resultado = resultado.replace(
                  "🔥 BONIFICAÇÃO:",
                  TEXTO_CLUBE_AZUL + "🔥 BONIFICAÇÃO:"
              )
          elif cia and "smiles" in cia.lower():
              resultado = resultado.replace(
                  "🔥 BONIFICAÇÃO:",
                  TEXTO_CLUBE_SMILES + "🔥 BONIFICAÇÃO:"
              )

          # ── Monta a chave anti-duplicata (dentro da mesma execução) ──
          parceiro_key = parceiro or ""
          cia_key = cia or ""
          data_key = data_str or ""
          promocao_key = (parceiro_key + cia_key + data_key).replace(" ", "")

          cont = 0
          if promocao_key not in list_promocao_key and status_promo == "Dentro do prazo":
              list_promocao_key.append(promocao_key)

              # id_grupo_envio, INSTANCE_ID e API_KEY já foram definidos lá no topo
              # do notebook (respeitando a flag ENVIAR_PARA_GRUPO_TESTE) — não
              # precisa redefinir aqui, e assim essa mensagem também respeita a flag.

              url_ws = f"https://api.zapperapi.com/{INSTANCE_ID}/messages/text"
              headers_ws = {
                  "X-Api-Key": API_KEY,
                  "Content-Type": "application/json"
              }
              payload = {
                  "jid": id_grupo_envio,
                  "message": resultado,
              }

              # Aguardar 20 segundos
              time.sleep(20)
              response_ws = requests.post(url_ws, json=payload, headers=headers_ws)
              print("📨 Promoção enviada no grupo!")

              # ── Monta ID da promoção ──
              parceiro_slug = (parceiro or "").lower().replace(" ", "-")
              cia_slug = (cia or "").lower().replace(" ", "-")
              bonus_slug = str(bonus_num) if bonus_num else "0"
              data_slug = f"Valido ate {data_str}" if data_str else "sem-data"
              id_promocao = f"{parceiro_slug}-{cia_slug}-{bonus_slug}-{data_slug}"

              agora = datetime.datetime.now(ZoneInfo("America/Sao_Paulo"))
              data_hora_str = agora.strftime("%d/%m/%Y %H:%M:%S")
              data_hoje = agora.date()
              nome_dia = nome_dia_semana_pt(agora)
              nome_mes = nome_mes_pt(agora)

              flag_feriado = 1 if data_hoje in feriados else 0
              nome_feriado = feriados.get(data_hoje, "")

              flag_evento = 1 if data_hoje in eventos else 0
              nome_evento = eventos.get(data_hoje, "")

              texto_original = html_texto[:500].replace("\n", " ").strip()

              linha = [
                  data_hora_str,
                  agora.strftime("%d/%m/%Y"),
                  nome_dia,
                  nome_mes,
                  id_promocao,
                  parceiro or "",
                  cia or "",
                  bonus_str or "",
                  bonus_num or 0,
                  data_str or "",
                  flag_feriado,
                  nome_feriado,
                  flag_evento,
                  nome_evento,
                  texto_original,
                  link
              ]

              gravar_no_sheets(spreadsheet, linha)
          else:
              print("⚠️  Promoção duplicada e/ou fora do prazo — ignorada.")

✅ 36 feriados e 54 eventos carregados.
📋 31 links encontrados | 31 não vencidos | processando até 15 nesta execução.

Publicada em: 11/07/2026
Status: Dentro do prazo
📨 Promoção enviada no grupo!
✅ Linha gravada no Google Sheets!

Publicada em: 10/07/2026
Status: Fora do prazo
⚠️  Promoção duplicada e/ou fora do prazo — ignorada.

Publicada em: 09/07/2026
Status: Fora do prazo
⚠️  Promoção duplicada e/ou fora do prazo — ignorada.

Publicada em: 08/07/2026
Status: Fora do prazo
⚠️  Promoção duplicada e/ou fora do prazo — ignorada.

Publicada em: 08/07/2026
Status: Fora do prazo
⚠️  Promoção duplicada e/ou fora do prazo — ignorada.

Publicada em: 07/07/2026
Status: Fora do prazo
⚠️  Promoção duplicada e/ou fora do prazo — ignorada.

Publicada em: 07/07/2026
Status: Fora do prazo
⚠️  Promoção duplicada e/ou fora do prazo — ignorada.

Publicada em: 07/07/2026
Status: Fora do prazo
⚠️  Promoção duplicada e/ou fora do prazo — ignorada.

Publicada em: 06/07/2026
Status: Fora do prazo
⚠️  Prom